# Step 20 — module preservation across cohorts

**This is preservation, not significance.** A protein cluster's internal coherence is not a
significance test — significance needs something to be significant *about*, and here that can only
be the patients. The two questions live in different notebooks:

| question | where | statistic |
|---|---|---|
| does this module **exist** in another cohort? | **here** | `modulePreservation` Z-summary |
| does it **relate to patients**? | step 12 | module ↔ trait correlation, BH-corrected |
| does the **patient partition** replicate? | step 05 | prediction strength, bootstrap Jaccard |

WGCNA gives a correlation structure and a tree cut. Neither is a test, and nothing in steps 11–13
established that a module found in cohort A is anything but a feature of cohort A. Given that A, B
and C yield 52, 23 and 46 modules from a random split of the same 260 donors, that question is
overdue.

**Method.** `WGCNA::modulePreservation` — Langfelder *et al.* (2011), *PLoS Comput Biol* 7:e1001057.
A's modules are the reference; B and C are the test networks. The statistic permutes module
assignment to build a null, and the published bands are **Z > 10 strong, 2–10 weak, < 2 none**.

**Two reference modules matter for reading it.** `grey` is the unassigned bin and `gold` is a
randomly sampled module — **`gold` is the null.** A module whose Z does not clear `gold` is not
preserved beyond chance, whatever the absolute number looks like.

In [ ]:
source("../src/paths.R")
ensure_pkg("WGCNA")               # no conda build for osx-arm64; see src/paths.R
suppressMessages(library(WGCNA))
options(stringsAsFactors = FALSE)

N_PERM <- 200
GREY_BACKGROUND <- 400     # unassigned probes carried along so the null has room

W <- lapply(c("A","B","C"), function(s) readRDS(art("wgcna_%s.rds", s)))
names(W) <- c("A","B","C")
P <- readRDS(art("step12_panels.rds"))
keep <- P$D[["A all15"]]$keep
sel  <- unlist(lapply(keep, function(k) colnames(W$A$X)[W$A$mods == k]))
set.seed(42)
grey <- sample(colnames(W$A$X)[W$A$mods == "grey"], GREY_BACKGROUND)
use  <- c(sel, grey)
cat(sprintf("reference: cohort A, %d modules, %s, plus %d unassigned probes as background\n",
            length(keep), size_str(sel), length(grey)))

multiData  <- lapply(c("A","B","C"), function(s) list(data = as.data.frame(W[[s]]$X[, use])))
names(multiData) <- c("A","B","C")
multiColor <- list(A = setNames(W$A$mods[match(use, colnames(W$A$X))], use))

t0 <- Sys.time()
# savePermutedStatistics = FALSE: the default writes a permutedStats-*.RData
# cache into the working directory, i.e. into ipynb/, where it is neither an
# artifact nor a notebook.
mp <- modulePreservation(multiData, multiColor, referenceNetworks = 1,
                         nPermutations = N_PERM, networkType = "signed",
                         randomSeed = 42, verbose = 0,
                         savePermutedStatistics = FALSE)
cat(sprintf("%d permutations, 2 test cohorts: %.0f s\n\n",
            N_PERM, as.numeric(difftime(Sys.time(), t0, units = "secs"))))

band <- function(z) ifelse(z > 10, "strong", ifelse(z > 2, "weak", "none"))
pres <- do.call(rbind, lapply(c("B","C"), function(tst) {
  z <- mp$preservation$Z[[1]][[which(names(multiData) == tst)]]
  o <- mp$preservation$observed[[1]][[which(names(multiData) == tst)]]
  data.frame(test = tst, module = rownames(z), size = o$moduleSize,
             Zsummary = round(z$Zsummary.pres, 1),
             medianRank = round(mp$preservation$observed[[1]][[
               which(names(multiData) == tst)]]$medianRank.pres, 1),
             band = band(z$Zsummary.pres))
}))
gold <- pres[pres$module == "gold", c("test","Zsummary")]
for (tst in c("B","C")) {
  g <- gold$Zsummary[gold$test == tst]
  cat(sprintf("--- A -> %s   (gold, the random-module null, Z = %.1f) ---\n", tst, g))
  p <- pres[pres$test == tst & !pres$module %in% c("gold","grey"), ]
  p$above_null <- p$Zsummary > g
  print(p[order(-p$Zsummary), c("module","size","Zsummary","medianRank","band","above_null")],
        row.names = FALSE)
  cat("\n")
}
saveRDS(list(pres = pres, n_perm = N_PERM, reference = "A", proteins = use),
        art("step20_preservation.rds"))

## Reading it

**Z-summary against the published bands is the headline, but `gold` is the honest floor.** `gold`
is a module of randomly chosen proteins, so it is what a non-module scores. Any module not clearly
above it has not been shown to be preserved, regardless of whether its absolute Z clears 10.

**What preservation does and does not say.** A preserved module means the *protein co-expression
structure* holds in the other cohort — those proteins still move together there. It says nothing
about whether the module relates to patient phenotype in that cohort; step 12 does that, and the
two can disagree. Cohort B is the case in point: its modules can be preserved while B's own
trait testing finds almost nothing, because preservation is about proteins and significance is
about patients.

**The obvious tension to look for.** Cohort B produced only 23 modules of its own and a single
trait-associated one. If A's modules are nonetheless well preserved in B, then the difference
between the cohorts is in the *tree cut*, not in the underlying correlation structure — which
would make the 52-vs-23 module count a property of `blockwiseModules` and its parameters rather
than of the biology. That is a checkable claim and this table is where to check it.